In [1]:
from pathlib import Path

PROJECT_ROOT = Path("..")
MODEL_RESULTS = PROJECT_ROOT / "model_results"
GENERATED_RESULTS = PROJECT_ROOT / "generated_results"
GENERATED_RESULTS.mkdir(exist_ok=True)

In [2]:
from collections import Counter
from pathlib import Path


def find_resamples(model_results, complete_only=False):
    """Resample indices found under model_results/*/Predictions/*/.

    complete_only=True keeps only those present for every classifier x dataset pair.
    """
    counts, pairs = Counter(), set()
    for f in Path(model_results).glob("*/Predictions/*/testResample*.csv"):
        idx = f.stem.removeprefix("testResample")
        if idx.isdigit():
            counts[int(idx)] += 1
            pairs.add((f.parents[2].name, f.parent.name))
    if complete_only:
        return sorted(r for r, n in counts.items() if n == len(pairs))
    return sorted(counts)


In [3]:
# Auto-discover classifiers and common datasets from disk
classifiers = sorted(p.name for p in MODEL_RESULTS.iterdir() if p.is_dir())
print("Classifiers found:", classifiers)

# dataset_sets = [
#     set(p.name for p in (MODEL_RESULTS / c / "Predictions").iterdir() if p.is_dir())
#     for c in classifiers
# ]
# datasets = sorted(set.intersection(*dataset_sets))
from aeon.datasets.tsc_datasets import univariate_equal_length

datasets = univariate_equal_length
print(f"Common datasets: {len(datasets)}")

# Detect available resamples from any classifier/dataset combo
# sample_files = list((MODEL_RESULTS / classifiers[0] / "Predictions" / datasets[0]).glob("testResample*.csv"))
# resamples = sorted(int(f.stem.replace("testResample", "")) for f in sample_files)
resamples = find_resamples(MODEL_RESULTS)
# resamples = [0,1,2,3,4,5]
print(f"Resamples: {resamples}")

Classifiers found: ['Catch22', 'FreshPRINCE', 'HIVECOTEV2', 'MultiRocketHydra', 'QUANT', 'RDST', 'ROCKET', 'TSCGlueEnhancedV4-Best-LogLoss-GPU', 'TSCGlueEnhancedV4-High-Accuracy-GPU', 'TSCGlueEnhancedV4-High-LogLoss-GPU', 'TSCGlueEnhancedV4-Low-Accuracy-GPU', 'TSCGlueEnhancedV4-Low-LogLoss-GPU', 'TSCGlueEnhancedV4-Medium-Accuracy-GPU', 'TSCGlueEnhancedV4-Medium-LogLoss-GPU', 'WEASEL_V2']


Common datasets: 112
Resamples: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29]


In [3]:
# Display names used by tsml_eval (figures, CSVs). Classifiers not listed keep their directory name.
RENAME = {
    # "HIVECOTEV2": "HC2",
}
_metric_map = {"Accuracy": "accuracy", "F1": "f1", "LogLoss": "log_loss", "ROCAUC": "roc_auc"}
for c in classifiers:
    if c.startswith("TSCGlueEnhancedV4-") and c.endswith("-GPU"):
        preset, metric = c.removeprefix("TSCGlueEnhancedV4-").removesuffix("-GPU").split("-")
        RENAME.setdefault(c, f"TSCGlue-{preset.lower()}-{_metric_map[metric]}")

# Columns of tsml_eval's per-metric CSVs are always the sorted display names.
EVAL_NAMES = sorted(RENAME.get(c, c) for c in classifiers)
RENAME

NameError: name 'classifiers' is not defined

In [ ]:
# classifiers.remove('TSCGlueEnhancedV4-Low-LogLoss-GPU')
# classifiers.remove('HIVECOTEV2')


# Count missing combinations (classifier x dataset x resample)
total = len(classifiers) * len(datasets) * len(resamples)
print(total, len(classifiers), len(datasets), len(resamples))
missing = 0
missing_list = []
for c in classifiers:
    for d in datasets:
        for r in resamples:
            path = MODEL_RESULTS / c / "Predictions" / d / f"testResample{r}.csv"
            if not path.exists():
                missing += 1
                missing_list.append((c, d, r))

print(f"Total combinations : {total}")
print(f"Present            : {total - missing}")
print(f"Missing            : {missing}")
per_model = len(datasets) * len(resamples)
print(f"\nMissing per model (out of {per_model}):")
for c in classifiers:
    n = sum(1 for mc, _, _ in missing_list if mc == c)
    if n:
        print(f"  {c:<35} {n}")

if missing_list:
    print("\nFirst 40 missing:")
    for c, d, r in missing_list[:40]:
        print(f"  {c} / {d} / resample {r}")

50400 15 112 30
Total combinations : 50400
Present            : 42742
Missing            : 7658

Missing per model (out of 3360):
  FreshPRINCE                         194
  HIVECOTEV2                          510
  TSCGlueEnhancedV4-Best-LogLoss-GPU  1568
  TSCGlueEnhancedV4-High-Accuracy-GPU 1344
  TSCGlueEnhancedV4-High-LogLoss-GPU  1568
  TSCGlueEnhancedV4-Low-LogLoss-GPU   762
  TSCGlueEnhancedV4-Medium-Accuracy-GPU 830
  TSCGlueEnhancedV4-Medium-LogLoss-GPU 882

First 40 missing:
  FreshPRINCE / ACSF1 / resample 17
  FreshPRINCE / ACSF1 / resample 27
  FreshPRINCE / ArrowHead / resample 17
  FreshPRINCE / ArrowHead / resample 22
  FreshPRINCE / ArrowHead / resample 27
  FreshPRINCE / Beef / resample 22
  FreshPRINCE / Beef / resample 27
  FreshPRINCE / BeetleFly / resample 22
  FreshPRINCE / BirdChicken / resample 29
  FreshPRINCE / BME / resample 22
  FreshPRINCE / BME / resample 27
  FreshPRINCE / BME / resample 28
  FreshPRINCE / Car / resample 17
  FreshPRINCE / CBF / resampl

In [ ]:
from tsml_eval.evaluation import evaluate_classifiers_by_problem

evaluate_classifiers_by_problem(
    str(MODEL_RESULTS),
    [(c, RENAME.get(c, c)) for c in classifiers],
    datasets,
    str(GENERATED_RESULTS),
    resamples=resamples,
    error_on_missing=False,
    eval_name="tscbench",
    verbose=True,
    continue_on_missing=False,
)

Loaded Catch22 on ACSF1 test resample 0.
Loaded Catch22 on ACSF1 test resample 1.
Loaded Catch22 on ACSF1 test resample 2.
Loaded Catch22 on ACSF1 test resample 3.
Loaded Catch22 on ACSF1 test resample 4.
Loaded Catch22 on ACSF1 test resample 5.
Loaded Catch22 on ACSF1 test resample 6.
Loaded Catch22 on ACSF1 test resample 7.
Loaded Catch22 on ACSF1 test resample 8.
Loaded Catch22 on ACSF1 test resample 9.
Loaded Catch22 on ACSF1 test resample 10.
Loaded Catch22 on ACSF1 test resample 11.
Loaded Catch22 on ACSF1 test resample 12.
Loaded Catch22 on ACSF1 test resample 13.
Loaded Catch22 on ACSF1 test resample 14.
Loaded Catch22 on ACSF1 test resample 15.
Loaded Catch22 on ACSF1 test resample 16.
Loaded Catch22 on ACSF1 test resample 17.
Loaded Catch22 on ACSF1 test resample 18.
Loaded Catch22 on ACSF1 test resample 19.
Loaded Catch22 on ACSF1 test resample 20.
Loaded Catch22 on ACSF1 test resample 21.
Loaded Catch22 on ACSF1 test resample 22.
Loaded Catch22 on ACSF1 test resample 23.
Lo

In [ ]:
# Display all generated figures inline
from IPython.display import display, Image

for fig_path in sorted(GENERATED_RESULTS.rglob("*.pdf")):
    print(fig_path.relative_to(PROJECT_ROOT))

for fig_path in sorted(GENERATED_RESULTS.rglob("*.png")):
    print(fig_path.relative_to(PROJECT_ROOT))
    display(Image(filename=str(fig_path)))

In [ ]:
import pandas as pd

for csv_path in sorted(GENERATED_RESULTS.glob("**/*summary*.csv")):
    print(csv_path.name)
    display(pd.read_csv(csv_path, index_col=0))

In [ ]:
import pickle
import matplotlib.pyplot as plt
import pandas as pd

for label, subdir, metric in [
    ("Accuracy", "Accuracy", "accuracy"),
    ("LogLoss", "LogLoss", "logloss"),
    ("F1", "F1", "f1"),
    ("ROCAUC", "AUROC", "auroc"),
]:
    fig_dir = GENERATED_RESULTS / "tscbench" / subdir
    pkl_path = fig_dir / "figures" / f"tscbench_{metric}_critical_difference.pickle"
    ranks_path = fig_dir / f"{metric}_ranks.csv"

    if not pkl_path.exists():
        print(f"{label}: no figure found at {pkl_path} (evaluate_classifiers_by_problem "
              "may have skipped it — check for missing results above)")
        continue

    n_datasets = pd.read_csv(ranks_path, index_col=0, header=None, skiprows=1).dropna().shape[0]
    print(f"{label}: {n_datasets} datasets")

    with open(pkl_path, "rb") as f:
        fig = pickle.load(f)
    display(fig)
    plt.close(fig)

In [ ]:
from aeon.visualisation import plot_critical_difference

# shown: metric name put on the TSCGlue labels. F1/ROCAUC plot the accuracy-tuned
# runs, relabelled as the metric being plotted.
for label, subdir, metric, shown in [
    ("Accuracy", "Accuracy", "accuracy", "accuracy"),
    ("LogLoss", "LogLoss", "logloss", "log_loss"),
    ("F1", "F1", "f1", "f1"),
    ("ROCAUC", "AUROC", "auroc", "roc_auc"),
]:
    variant = "log_loss" if metric == "logloss" else "accuracy"
    means = pd.read_csv(
        GENERATED_RESULTS / "tscbench" / subdir / f"{metric}_mean.csv",
        index_col=0, header=None, skiprows=1,
    ).set_axis(EVAL_NAMES, axis=1)
    cols = [c for c in means.columns if not c.startswith("TSCGlue-") or c.endswith(f"-{variant}")]
    means = means[cols].dropna()
    labels = [c.removesuffix(f"-{variant}") + f"-{shown}" if c.startswith("TSCGlue-") else c for c in cols]
    print(f"{label} (TSCGlue {variant}): {means.shape[0]} datasets")

    fig, ax = plot_critical_difference(
        means.values, labels, lower_better=(metric == "logloss")
    )
    display(fig)
    plt.close(fig)
